# MedGemma-4B server on a free Colab T4 (llama.cpp + ngrok)

Why not vLLM here: a T4 (compute 7.5) has no bfloat16. vLLM won't run Gemma 3 in float16 because its
activations overflow, and float32 weights for the 4B multimodal model (~17 GB) don't fit in 15 GB.
llama.cpp runs Gemma 3 correctly on a T4 from a GGUF file, and its server exposes the same
OpenAI-style `/v1/completions`, `/v1/chat/completions`, `/health` and `--api-key`, so the local app's `.env` format stays the same.

Run top to bottom. No session restart is needed.

## 1. Confirm the GPU

In [1]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
import torch
assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU."
print("GPU:", torch.cuda.get_device_name(0))

name, memory.total [MiB], compute_cap
Tesla T4, 15360 MiB, 7.5
GPU: Tesla T4


## 2. Secrets and settings

Put `HF_TOKEN`, `NGROK_TOKEN` and (optionally) `COLAB_API_KEY` in Colab's **Secrets** panel (key icon in the
left sidebar) and grant this notebook access. Don't paste tokens into cells: anyone you share the notebook with gets them.
If `COLAB_API_KEY` is not set, a random key is generated and printed with the tunnel URL.

In [2]:
!pip install -q pyngrok
import os

HF_TOKEN    = "hf_YOUR_HUGGINGFACE_TOKEN_HERE"
NGROK_TOKEN = "YOUR_NGROK_AUTHTOKEN_HERE"

API_KEY    = "change-me-please"                # must equal COLAB_API_KEY in .env

MODEL_ID   = "google/medgemma-4b-it"           # must equal MODEL_ID in .env
GGUF_REPO  = "unsloth/medgemma-4b-it-GGUF"     # GGUF conversion of the same weights
QUANT      = "Q8_0"                            # ~4 GB, near-lossless; "Q4_K_M" is smaller/faster
MAX_LEN    = 8192                              # must equal MAX_MODEL_LEN in .env
PORT       = 8000

os.environ["HF_TOKEN"] = HF_TOKEN
from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## 3. Build llama-server for the T4

Compiles only for the T4's architecture (sm_75), so it takes about 5-10 minutes on Colab's CPUs.
Set `USE_DRIVE_CACHE = True` to save the binary to Google Drive; later sessions then copy it back in seconds.

In [3]:
import os, shutil, subprocess

USE_DRIVE_CACHE = True
BIN       = "/content/llama-server"
DRIVE_BIN = "/content/drive/MyDrive/colab-cache/llama-server-sm75"

if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount("/content/drive")
    if os.path.exists(DRIVE_BIN) and not os.path.exists(BIN):
        shutil.copy(DRIVE_BIN, BIN); os.chmod(BIN, 0o755)
        print("Restored llama-server from Drive.")

if not os.path.exists(BIN):
    def run(c): print("$", c); subprocess.run(c, shell=True, check=True)
    run("rm -rf /content/llama.cpp && git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp")
    run("cmake -S /content/llama.cpp -B /content/llama.cpp/build "
        "-DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 "
        "-DBUILD_SHARED_LIBS=OFF -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release")
    run("cmake --build /content/llama.cpp/build --config Release -j $(nproc) --target llama-server")
    shutil.copy("/content/llama.cpp/build/bin/llama-server", BIN); os.chmod(BIN, 0o755)
    if USE_DRIVE_CACHE:
        os.makedirs(os.path.dirname(DRIVE_BIN), exist_ok=True)
        shutil.copy(BIN, DRIVE_BIN); print("Cached to Drive.")

subprocess.run([BIN, "--version"])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


CompletedProcess(args=['/content/llama-server', '--version'], returncode=0)

In [4]:
from google.colab import drive
import os, shutil
drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/colab-cache", exist_ok=True)
shutil.copy("/content/llama-server", "/content/drive/MyDrive/colab-cache/llama-server-sm75")
print("Saved.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved.


## 4. Download the GGUF

In [5]:
from huggingface_hub import list_repo_files, hf_hub_download

files = list_repo_files(GGUF_REPO)
matches = [f for f in files if f.endswith(".gguf") and QUANT.lower() in f.lower() and "mmproj" not in f.lower()]
assert matches, f"No {QUANT} file in {GGUF_REPO}. Available: {[f for f in files if f.endswith('.gguf')]}"
GGUF_PATH = hf_hub_download(GGUF_REPO, matches[0])
print("Model file:", GGUF_PATH, f"({os.path.getsize(GGUF_PATH)/1e9:.1f} GB)")

Model file: /root/.cache/huggingface/hub/models--unsloth--medgemma-4b-it-GGUF/snapshots/fa17ea2647ef236db91adfe1465a61ab6562cf2c/medgemma-4b-it-Q8_0.gguf (4.1 GB)


## 5. Start the server

| flag | why |
|---|---|
| `-ngl 99` | put every layer on the GPU |
| `-c` | context length, same as the old `MAX_LEN` |
| `--jinja` | use the model's own Gemma chat template for `/v1/chat/completions` |
| `--alias` | the server answers to `google/medgemma-4b-it`, so requests from the app don't need changing |
| `--repeat-penalty 1.1` | server-side default, since llama.cpp names this parameter differently from vLLM |

Prompt caching is on by default: a follow-up turn that re-sends the same system block and patient record skips re-processing that prefix.

In [6]:
import subprocess, time, requests

LOG = "/content/llama-server.log"
cmd = [
    BIN, "-m", GGUF_PATH,
    "--alias", MODEL_ID,
    "-ngl", "99",
    "-c", str(MAX_LEN),
    "--jinja",
    "--repeat-penalty", "1.1",
    "--host", "0.0.0.0", "--port", str(PORT),
    "--api-key", API_KEY,
]
logfile = open(LOG, "w")
server = subprocess.Popen(cmd, stdout=logfile, stderr=subprocess.STDOUT)

start, i = time.time(), 0
while True:
    if server.poll() is not None:
        print("".join(open(LOG).readlines()[-40:]))
        raise SystemExit("llama-server failed to start.")
    try:
        if requests.get(f"http://127.0.0.1:{PORT}/health", timeout=2).status_code == 200:
            print(f"Ready in {time.time()-start:.0f}s."); break
    except requests.RequestException:
        pass
    if i % 4 == 0:
        tail = open(LOG).readlines()[-1:]
        if tail: print("  ", tail[0].strip()[:140])
    i += 1; time.sleep(5)

!grep -E "offloaded|CUDA0 model buffer" {LOG} | tail -3

Ready in 5s.


## 6. Open the tunnel

Copy the printed block into `meditron-local/.env`. The free ngrok URL changes every session.

In [7]:
from pyngrok import ngrok
ngrok.set_auth_token(NGROK_TOKEN)
ngrok.kill()
public_url = ngrok.connect(PORT, "http").public_url.replace("http://", "https://")

print("Paste this into meditron-local/.env:\n")
print(f"COLAB_API_BASE={public_url}")
print(f"COLAB_API_KEY={API_KEY}")
print(f"MODEL_ID={MODEL_ID}")
print(f"MAX_MODEL_LEN={MAX_LEN}")

Paste this into meditron-local/.env:

COLAB_API_BASE=https://chalcographic-claudette-solemn.ngrok-free.dev
COLAB_API_KEY=change-me-please
MODEL_ID=google/medgemma-4b-it
MAX_MODEL_LEN=8192


## 7. Test

MedGemma is a Gemma 3 model, so it expects Gemma turns (`<start_of_turn>user` / `<start_of_turn>model`),
not ChatML (`<|im_start|>`). Gemma has no system role; the system text goes at the top of the first user turn.

- **A** is the easiest path for the app: `/v1/chat/completions` with a messages list, and the server applies the template.
- **B** shows the raw format, if the app's `memory.py` keeps building prompts itself for `/v1/completions`.

In [8]:
import requests
H = {"Authorization": f"Bearer {API_KEY}"}
SYSTEM = ("You are a careful medical information assistant. Answer in plain language.\n\n"
          "PATIENT RECORD\nAge: 31\nExisting conditions: mild asthma\nAllergies: penicillin")

# A. chat endpoint
r = requests.post(f"http://127.0.0.1:{PORT}/v1/chat/completions", headers=H, timeout=180, json={
    "model": MODEL_ID, "max_tokens": 300, "temperature": 0.3, "top_p": 0.9,
    "messages": [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": "I've had a dry cough for four days and a mild sore throat. No fever."},
        {"role": "assistant", "content": "A dry cough with a mild sore throat and no fever most often follows a viral upper respiratory infection, and it usually settles over one to two weeks."},
        {"role": "user", "content": "My asthma inhaler helps a little but the cough is worse at night. Should I be worried?"},
    ]})
j = r.json(); print("A:", j["choices"][0]["message"]["content"].strip()); print("usage:", j["usage"], "\n")

# B. raw Gemma-format prompt on /v1/completions (BOS is added by the server)
prompt = (
    f"<start_of_turn>user\n{SYSTEM}\n\nI've had a dry cough for four days and a mild sore throat. No fever.<end_of_turn>\n"
    "<start_of_turn>model\nA dry cough with a mild sore throat and no fever most often follows a viral upper respiratory infection, and it usually settles over one to two weeks.<end_of_turn>\n"
    "<start_of_turn>user\nMy asthma inhaler helps a little but the cough is worse at night. Should I be worried?<end_of_turn>\n"
    "<start_of_turn>model\n"
)
r = requests.post(f"http://127.0.0.1:{PORT}/v1/completions", headers=H, timeout=180, json={
    "model": MODEL_ID, "prompt": prompt, "max_tokens": 300, "temperature": 0.3, "top_p": 0.9,
    "stop": ["<end_of_turn>", "<start_of_turn>"]})
j = r.json(); print("B:", j["choices"][0]["text"].strip()); print("usage:", j["usage"])

A: It's understandable that you're concerned about your cough worsening at night with your asthma inhaler providing only slight relief. Here's what we can consider:

*   **Asthma Control:** Even if your inhaler helps a little, it might not be fully controlling the inflammation in your airways that's causing the cough and nighttime symptoms.
*   **Nighttime Cough:** Nighttime cough is common with asthma due to several factors like lying down which allows mucus to pool in the throat or increased sensitivity to allergens in the bedroom.
*   **Possible Causes:** The cough could be related to:

    *   **Asthma flare-up:** Even a mild one can cause coughing, especially at night.
    *   **Allergens:** Dust mites, pet dander, or mold in your bedroom might be triggering your asthma and cough.
    *   **Postnasal drip:** Mucus dripping down the back of your throat can irritate it and cause a cough.

Given this information, here's what I recommend:

1.  **Contact Your Doctor:** It is important 

## 8. Keep the session awake

Leave this running and the tab open while you use the app. Interrupt it when done.

In [ ]:
import time, requests
from datetime import datetime
while True:
    try:
        ok = requests.get(f"http://127.0.0.1:{PORT}/health", timeout=5).status_code == 200
    except requests.RequestException:
        ok = False
    print(f"{datetime.now():%H:%M:%S}  {'serving' if ok else 'DOWN - re-run cell 5'}   ", end="\r")
    time.sleep(60)

## 9. Shut down

In [ ]:
from pyngrok import ngrok
ngrok.kill()
try:
    server.terminate(); server.wait(timeout=30)
except Exception as exc:
    print(exc)
print("Tunnel closed, server stopped.")